# Strategy B: Direct Sparse Drift With Lyapunov Feasibility on GOLDEN

This notebook prototypes the sparsity-preserving Lyapunov route. It samples `A` directly on the GOLDEN dynamic topology, keeps absent direct edges exactly zero, and filters draws by an explicit Lyapunov certificate. This is a prior-level benchmark only.

In [1]:

from __future__ import annotations

import json
import time
from pathlib import Path

import networkx as nx
import numpy as np
from scipy.linalg import expm, solve_continuous_lyapunov


def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    while current != current.parent:
        if (current / "apps" / "data-pipeline").exists() and (current / "data").exists():
            return current
        current = current.parent
    raise RuntimeError("Could not locate repository root from notebook working directory")


REPO_ROOT = find_repo_root(Path.cwd())
GOLDEN_RUN = REPO_ROOT / "data" / ".private" / "GOLDEN" / "run"
STAGE1B_PATH = GOLDEN_RUN / "stage-1b.json"
MEGAPROMPT_PATH = GOLDEN_RUN / "stage-4-megaprompt.json"

stage1b = json.loads(STAGE1B_PATH.read_text())
megaprompt = json.loads(MEGAPROMPT_PATH.read_text())
accepted = megaprompt["accepted"]
causal_spec = stage1b["causal_spec"]
resolved_priors = {
    prior["parameter"]: prior
    for prior in accepted["resolved_priors"]
    if prior is not None
}

all_state_names = list(causal_spec["estimation"]["state_order"])
constructs = {construct["name"]: construct for construct in causal_spec["latent"]["constructs"]}
latent_names = [
    name
    for name in all_state_names
    if constructs[name].get("role") == "endogenous"
    and constructs[name].get("temporal_status") == "time_varying"
]
n_latent = len(latent_names)
latent_index = {name: idx for idx, name in enumerate(latent_names)}

drift_mask = np.eye(n_latent, dtype=bool)
for edge in causal_spec["latent"]["edges"]:
    cause = edge["cause"]
    effect = edge["effect"]
    parameter = f"beta_{cause}_{effect}"
    if cause in latent_index and effect in latent_index and parameter in resolved_priors:
        drift_mask[latent_index[effect], latent_index[cause]] = True

diag_mask = np.eye(n_latent, dtype=bool)
allowed_offdiag_mask = drift_mask & ~diag_mask
structural_zero_mask = (~drift_mask) & ~diag_mask
allowed_positions = [
    (row, col)
    for row in range(n_latent)
    for col in range(n_latent)
    if row != col and drift_mask[row, col]
]


def duration_to_days(value: str) -> float:
    if value.endswith("d"):
        return float(value[:-1])
    if value.endswith("h"):
        return float(value[:-1]) / 24.0
    raise ValueError(f"Unsupported model clock {value!r}")


model_dt_days = duration_to_days(causal_spec["measurement"]["model_clock"])

G = nx.DiGraph()
G.add_nodes_from(range(n_latent))
for row, col in allowed_positions:
    G.add_edge(col, row)
sccs = [tuple(sorted(component)) for component in nx.strongly_connected_components(G)]
sccs = sorted(sccs, key=lambda component: min(component))
component_index = {
    node: comp_idx
    for comp_idx, component in enumerate(sccs)
    for node in component
}
condensed = nx.DiGraph()
condensed.add_nodes_from(range(len(sccs)))
for row, col in allowed_positions:
    source_comp = component_index[col]
    target_comp = component_index[row]
    if source_comp != target_comp:
        condensed.add_edge(source_comp, target_comp)

macro_allowed_mask = np.eye(n_latent, dtype=bool)
for row in range(n_latent):
    for col in range(n_latent):
        if row == col:
            continue
        source_comp = component_index[col]
        target_comp = component_index[row]
        if source_comp == target_comp or condensed.has_edge(source_comp, target_comp):
            macro_allowed_mask[row, col] = True
macro_zero_mask = (~macro_allowed_mask) & ~diag_mask

print(f"Golden run: {GOLDEN_RUN.relative_to(REPO_ROOT)}")
print(f"Dynamic drift block ({n_latent} states): {', '.join(latent_names)}")
print("Excluded retained states:")
for name in all_state_names:
    if name not in latent_index:
        construct = constructs[name]
        print(f"  {name}: {construct.get('role')} / {construct.get('temporal_status')}")
print(f"Model interval: {model_dt_days:g} day")
print(f"Allowed off-diagonal dynamic drift entries: {int(allowed_offdiag_mask.sum())}")
print(f"Structural-zero off-diagonal dynamic entries: {int(structural_zero_mask.sum())}")
print("SCCs:")
for comp_idx, component in enumerate(sccs):
    labels = [latent_names[idx] for idx in component]
    print(f"  C{comp_idx}: {labels}")
print("Allowed dynamic topology edges:")
for row, col in allowed_positions:
    print(f"  {latent_names[col]} -> {latent_names[row]}")


Golden run: data/.private/GOLDEN/run
Dynamic drift block (9 states): sleep_quality, sleep_duration, screen_time, evening_screen_use, screen_content_type, social_media_use, stress, mental_health, bedtime_delay
Excluded retained states:
  chronotype: exogenous / time_invariant
Model interval: 1 day
Allowed off-diagonal dynamic drift entries: 11
Structural-zero off-diagonal dynamic entries: 61
SCCs:
  C0: ['sleep_quality']
  C1: ['sleep_duration']
  C2: ['screen_time']
  C3: ['evening_screen_use']
  C4: ['screen_content_type']
  C5: ['social_media_use']
  C6: ['stress', 'mental_health']
  C7: ['bedtime_delay']
Allowed dynamic topology edges:
  sleep_duration -> sleep_quality
  stress -> sleep_quality
  mental_health -> sleep_quality
  bedtime_delay -> sleep_duration
  stress -> screen_time
  mental_health -> screen_time
  screen_time -> evening_screen_use
  screen_content_type -> social_media_use
  mental_health -> stress
  stress -> mental_health
  evening_screen_use -> bedtime_delay


## Construction

A sparse candidate `A` is sampled from the accepted GOLDEN scalar priors. A draw is feasible when there exists `P > 0` such that `A.T @ P + P @ A < 0`. For a stable candidate, the continuous Lyapunov equation `A.T @ P + P @ A = -I` gives a certificate. This notebook uses that certificate as a rejection filter, which is exactly the operational cost of this design.

In [2]:

def sample_prior_1d(prior: dict, rng: np.random.Generator, n_draws: int) -> np.ndarray:
    family = prior["distribution"]
    params = prior["params"]
    if family == "Beta":
        return rng.beta(params["alpha"], params["beta"], size=n_draws)
    if family == "Uniform":
        return rng.uniform(params["lower"], params["upper"], size=n_draws)
    if family == "Normal":
        return rng.normal(params["mu"], params["sigma"], size=n_draws)
    raise ValueError(f"Unsupported drift prior family {family!r}")


def sample_current_scalar_prior(rng: np.random.Generator, n_draws: int) -> np.ndarray:
    draws = np.zeros((n_draws, n_latent, n_latent), dtype=float)
    for idx, name in enumerate(latent_names):
        rho = sample_prior_1d(resolved_priors[f"rho_{name}"], rng, n_draws)
        rho = np.clip(rho, 1e-8, 1.0 - 1e-8)
        draws[:, idx, idx] = np.log(rho) / model_dt_days
    for row, col in allowed_positions:
        beta = sample_prior_1d(
            resolved_priors[f"beta_{latent_names[col]}_{latent_names[row]}"],
            rng,
            n_draws,
        )
        draws[:, row, col] = beta / model_dt_days
    return draws


def interval_response_samples(draws: np.ndarray, dt_days: float, max_draws: int = 1000) -> np.ndarray:
    n = min(max_draws, draws.shape[0])
    return np.stack([expm(draws[i] * dt_days) for i in range(n)], axis=0)


def summarize_drift_samples(name: str, draws: np.ndarray, elapsed_seconds: float) -> dict[str, float | str]:
    eigvals = np.linalg.eigvals(draws)
    max_real = eigvals.real.max(axis=1)
    direct_absent_abs = np.abs(draws[:, structural_zero_mask])
    macro_absent_abs = np.abs(draws[:, macro_zero_mask])
    allowed_abs = np.abs(draws[:, allowed_offdiag_mask])
    response = interval_response_samples(draws, model_dt_days)
    direct_absent_response_abs = np.abs(response[:, structural_zero_mask])
    allowed_response_abs = np.abs(response[:, allowed_offdiag_mask])
    return {
        "name": name,
        "draws": draws.shape[0],
        "stable_rate": float(np.mean(max_real < 0.0)),
        "margin_q05": float(np.quantile(-max_real, 0.05)),
        "diag_mean": float(np.mean(np.diagonal(draws, axis1=1, axis2=2))),
        "direct_absent_a_q90": float(np.quantile(direct_absent_abs, 0.90)),
        "macro_absent_a_max": float(np.max(macro_absent_abs)),
        "allowed_a_q90": float(np.quantile(allowed_abs, 0.90)),
        "direct_absent_response_q90": float(np.quantile(direct_absent_response_abs, 0.90)),
        "allowed_response_q90": float(np.quantile(allowed_response_abs, 0.90)),
        "seconds": float(elapsed_seconds),
    }


def print_summary_table(rows: list[dict[str, float | str]]) -> None:
    columns = [
        ("name", "prior"),
        ("draws", "draws"),
        ("stable_rate", "stable"),
        ("margin_q05", "margin q05"),
        ("diag_mean", "diag mean"),
        ("direct_absent_a_q90", "direct absent |A| q90"),
        ("macro_absent_a_max", "macro absent |A| max"),
        ("allowed_a_q90", "allowed |A| q90"),
        ("direct_absent_response_q90", "direct absent |exp(AΔ)| q90"),
        ("allowed_response_q90", "allowed |exp(AΔ)| q90"),
        ("seconds", "seconds"),
    ]
    widths = []
    for key, label in columns:
        values = [label]
        for row in rows:
            value = row[key]
            values.append(str(value) if isinstance(value, str) else f"{value:.4g}")
        widths.append(max(len(v) for v in values))
    header = "  ".join(label.ljust(width) for (_, label), width in zip(columns, widths))
    print(header)
    print("  ".join("-" * width for width in widths))
    for row in rows:
        parts = []
        for (key, _label), width in zip(columns, widths):
            value = row[key]
            text = str(value) if isinstance(value, str) else f"{value:.4g}"
            parts.append(text.ljust(width))
        print("  ".join(parts))


In [3]:

def lyapunov_certificate(
    drift: np.ndarray,
    *,
    spectral_margin: float = 1e-5,
    p_min_eig: float = 1e-8,
    lyapunov_max_eig: float = -0.5,
) -> dict[str, float | bool]:
    eig_max = float(np.max(np.linalg.eigvals(drift).real))
    if eig_max >= -spectral_margin:
        return {
            "feasible": False,
            "max_real_eig": eig_max,
            "p_min_eig": float("nan"),
            "lyapunov_max_eig": float("nan"),
        }

    p = solve_continuous_lyapunov(drift.T, -np.eye(drift.shape[0]))
    p = 0.5 * (p + p.T)
    p_eig_min = float(np.min(np.linalg.eigvalsh(p)))
    lyapunov = drift.T @ p + p @ drift
    lyapunov_eig_max = float(np.max(np.linalg.eigvalsh(lyapunov)))
    feasible = p_eig_min > p_min_eig and lyapunov_eig_max < lyapunov_max_eig
    return {
        "feasible": feasible,
        "max_real_eig": eig_max,
        "p_min_eig": p_eig_min,
        "lyapunov_max_eig": lyapunov_eig_max,
    }


def lyapunov_filter(draws: np.ndarray) -> tuple[np.ndarray, list[dict[str, float | bool]]]:
    diagnostics = [lyapunov_certificate(draw) for draw in draws]
    keep = np.asarray([bool(item["feasible"]) for item in diagnostics], dtype=bool)
    return draws[keep], diagnostics


def sample_lyapunov_rejection_prior(
    rng: np.random.Generator,
    n_draws: int,
    *,
    batch_size: int = 1000,
    max_candidates: int = 20000,
) -> tuple[np.ndarray, dict[str, float]]:
    accepted_batches = []
    candidate_count = 0
    feasible_count = 0
    while feasible_count < n_draws and candidate_count < max_candidates:
        batch = sample_current_scalar_prior(rng, min(batch_size, max_candidates - candidate_count))
        candidate_count += batch.shape[0]
        feasible, _diagnostics = lyapunov_filter(batch)
        if feasible.size:
            accepted_batches.append(feasible)
            feasible_count += feasible.shape[0]

    if not accepted_batches:
        accepted = np.zeros((0, n_latent, n_latent), dtype=float)
    else:
        accepted = np.concatenate(accepted_batches, axis=0)[:n_draws]
    return accepted, {
        "candidates": float(candidate_count),
        "accepted": float(accepted.shape[0]),
        "acceptance_rate": float(accepted.shape[0] / candidate_count) if candidate_count else 0.0,
    }


def perturb_for_rejection_stress(draws: np.ndarray, *, decay_multiplier: float, effect_multiplier: float) -> np.ndarray:
    perturbed = draws.copy()
    perturbed[:, np.arange(n_latent), np.arange(n_latent)] *= decay_multiplier
    perturbed[:, allowed_offdiag_mask] *= effect_multiplier
    return perturbed


## Benchmark

In [4]:

N_DRAWS = 3000
rng = np.random.default_rng(20260429)

start = time.perf_counter()
scalar_draws = sample_current_scalar_prior(rng, N_DRAWS)
scalar_elapsed = time.perf_counter() - start

start = time.perf_counter()
lyapunov_draws, rejection_stats = sample_lyapunov_rejection_prior(rng, N_DRAWS)
lyapunov_elapsed = time.perf_counter() - start

rows = [
    summarize_drift_samples("accepted scalar prior", scalar_draws, scalar_elapsed),
    summarize_drift_samples("Lyapunov-filtered sparse", lyapunov_draws, lyapunov_elapsed),
]
print_summary_table(rows)
print("Rejection stats:", rejection_stats)


prior                     draws  stable  margin q05  diag mean  direct absent |A| q90  macro absent |A| max  allowed |A| q90  direct absent |exp(AΔ)| q90  allowed |exp(AΔ)| q90  seconds  
------------------------  -----  ------  ----------  ---------  ---------------------  --------------------  ---------------  ---------------------------  ---------------------  ---------
accepted scalar prior     3000   1       3.623       -4.744     0                      0                     1.5              0.003579                     0.01431                0.0009225
Lyapunov-filtered sparse  3000   1       3.582       -4.745     0                      0                     1.5              0.003542                     0.01437                0.2025   
Rejection stats: {'candidates': 3000.0, 'accepted': 3000.0, 'acceptance_rate': 1.0}


## Feasibility Diagnostics

In [5]:

_, baseline_diagnostics = lyapunov_filter(scalar_draws[:500])
print(
    "Baseline first-500 feasible:",
    sum(bool(item["feasible"]) for item in baseline_diagnostics),
    "/",
    len(baseline_diagnostics),
)
print(
    "Baseline min P eigenvalue q05:",
    np.quantile([item["p_min_eig"] for item in baseline_diagnostics], 0.05),
)

stress_candidates = perturb_for_rejection_stress(
    scalar_draws[:500],
    decay_multiplier=0.25,
    effect_multiplier=2.0,
)
stress_feasible, stress_diagnostics = lyapunov_filter(stress_candidates)
print(
    "Stress probe feasible:",
    stress_feasible.shape[0],
    "/",
    stress_candidates.shape[0],
)
print(
    "Stress probe stable rate:",
    np.mean([item["max_real_eig"] < 0.0 for item in stress_diagnostics]),
)


Baseline first-500 feasible: 500 / 500
Baseline min P eigenvalue q05: 0.06973046930882021
Stress probe feasible: 0 / 500
Stress probe stable rate: 0.0


## Reading

This route preserves direct topology exactly, but the prior no longer guarantees stable draws by construction. GOLDEN's accepted scalar priors happen to be strongly stable, so rejection is cheap here. The stress probe weakens the diagonals and doubles effects on the same topology; then the Lyapunov filter rejects every checked draw. That is the design risk: production would need rejection, projection, LMI solving, or constrained MCMC when the authored prior is less conservative.